# Module 06 — Snapshot Lifecycle Management (SLM)

SLM automates snapshot creation and retention on a cron schedule.
It is the production-grade alternative to manually calling `_snapshot` APIs.

## SLM concepts

```
SLM Policy
├── schedule      — cron expression (when to take snapshots)
├── name          — snapshot name template (supports date math)
├── repository    — where to write snapshots
├── config        — what to snapshot (indices, global state, feature states)
└── retention     — when to delete old snapshots
    ├── expire_after  — max age
    ├── min_count     — always keep at least N
    └── max_count     — never keep more than N
```

In [ ]:
import time
from helpers import *

es = get_client()
wait_for_green(es)
register_fs_repo(es)

---
## 1. Create an SLM Policy

In [ ]:
heading('1. Create SLM policy: training-hourly')

es.slm.put_lifecycle(
    policy_id='training-hourly',
    body={
        'schedule': '0 0 * * * ?',         # Every hour at minute 0
        'name': '<training-snap-{now/d}>',  # Date-math name
        'repository': FS_REPO_NAME,
        'config': {
            'indices': ['kibana_sample_data_*'],
            'include_global_state': True,
            'feature_states': ['kibana'],
            'ignore_unavailable': True,
            'partial': False,
            'metadata': {
                'policy': 'training-hourly',
                'environment': 'training',
            },
        },
        'retention': {
            'expire_after': '7d',   # delete snapshots older than 7 days
            'min_count': 3,         # always keep at least 3
            'max_count': 24,        # never keep more than 24
        },
    },
)
success('Policy training-hourly created')

In [ ]:
# Get policy details
policy = es.slm.get_lifecycle(policy_id='training-hourly')
pp(dict(policy['training-hourly']), 'training-hourly policy')

---
## 2. Execute a Policy Manually

In production, SLM runs on the schedule. In training, we trigger it immediately.

In [ ]:
heading('2. Execute policy immediately')

execute_resp = es.slm.execute_lifecycle(policy_id='training-hourly')
snap_name = execute_resp['snapshot_name']
success(f'Triggered snapshot: {snap_name}')

# Wait for it to complete
snap = wait_for_snapshot(es, FS_REPO_NAME, snap_name)
console.print(f'  State      : {snap["state"]}')
console.print(f'  Indices    : {snap["indices"]}')
console.print(f'  Feature states: {[fs["feature_name"] for fs in snap.get("feature_states", [])]}')

---
## 3. Inspect Policy Execution History

In [ ]:
heading('3. Policy execution history')

policy = es.slm.get_lifecycle(policy_id='training-hourly')
info_obj = policy['training-hourly']

last_success = info_obj.get('last_success')
last_failure = info_obj.get('last_failure')

if last_success:
    pp(last_success, 'last_success')
if last_failure:
    pp(last_failure, 'last_failure')
else:
    success('No failures recorded.')

# Also show next scheduled execution
next_exec = info_obj.get('next_execution')
next_exec_ms = info_obj.get('next_execution_millis')
console.print(f'  Next execution: {next_exec}')

---
## 4. SLM Statistics

In [ ]:
heading('4. SLM cluster-wide statistics')

stats = es.slm.get_stats()
pp(dict(stats), 'GET /_slm/stats')

info('Key fields:')
info('  retention_runs       — how many times retention has executed')
info('  retention_failed     — retention failures')
info('  retention_timed_out  — retention tasks that hit the timeout')
info('  retention_deletion_time — total time spent deleting snapshots')
info('  policy_results       — per-policy success/failure counts')

---
## 5. SLM Status — Start and Stop

In [ ]:
heading('5. SLM status')

status = es.slm.get_status()
pp(dict(status), 'GET /_slm/status')

In [ ]:
# Stop SLM (no new scheduled snapshots will run)
es.slm.stop()
time.sleep(1)
status = es.slm.get_status()
warn(f'SLM stopped: operation_mode={status["operation_mode"]}')

# Restart SLM
es.slm.start()
time.sleep(1)
status = es.slm.get_status()
success(f'SLM restarted: operation_mode={status["operation_mode"]}')

---
## 6. Execute Retention Manually

In [ ]:
heading('6. Execute retention manually')

# Create several snapshots so retention has something to work with
for i in range(3):
    execute_resp = es.slm.execute_lifecycle(policy_id='training-hourly')
    wait_for_snapshot(es, FS_REPO_NAME, execute_resp['snapshot_name'])
    info(f'Created snapshot {i+1}: {execute_resp["snapshot_name"]}')

# List all SLM-managed snapshots
all_snaps = es.snapshot.get(repository=FS_REPO_NAME, snapshot='training-snap-*')
console.print(f'  SLM-managed snapshots before retention: {len(all_snaps["snapshots"])}')
for s in all_snaps['snapshots']:
    console.print(f"    {s['snapshot']}  state={s['state']}")

# Run retention — since min_count=3 and max_count=24, nothing will be deleted yet
# But this shows the API
es.slm.execute_retention()
time.sleep(3)

after_snaps = es.snapshot.get(repository=FS_REPO_NAME, snapshot='training-snap-*')
console.print(f'  SLM-managed snapshots after retention: {len(after_snaps["snapshots"])}')

---
## 7. Multiple Policies — Different Schedules & Retention

A production strategy often uses multiple SLM policies:

| Policy | Schedule | Retention | Purpose |
|--------|----------|-----------|--------|
| `hourly` | `0 0 * * * ?` | 7d / min 3 / max 24 | Short-term, frequent |
| `daily` | `0 30 1 * * ?` | 30d / min 7 / max 30 | Daily rollup |
| `weekly` | `0 0 2 ? * MON` | 90d / min 4 / max 12 | Weekly archive |

In [ ]:
heading('7. Create daily and weekly SLM policies')

for policy_id, schedule, expire, min_c, max_c in [
    ('training-daily',  '0 30 1 * * ?',   '30d', 7,  30),
    ('training-weekly', '0 0 2 ? * MON',  '90d', 4,  12),
]:
    es.slm.put_lifecycle(
        policy_id=policy_id,
        body={
            'schedule': schedule,
            'name': f'<{policy_id}-{{now/d}}>',
            'repository': FS_REPO_NAME,
            'config': {
                'indices': ['kibana_sample_data_*'],
                'include_global_state': True,
                'feature_states': ['kibana'],
            },
            'retention': {
                'expire_after': expire,
                'min_count': min_c,
                'max_count': max_c,
            },
        },
    )
    success(f'Policy created: {policy_id}  schedule={schedule}  expire_after={expire}')

# List all policies
all_policies = es.slm.get_lifecycle()
t = Table('Policy', 'Schedule', 'Expire after', 'Min/Max count', 'Next execution')
for pid, pdata in all_policies.items():
    cfg = pdata.get('policy', {})
    ret = cfg.get('retention', {})
    t.add_row(
        pid,
        cfg.get('schedule', '—'),
        ret.get('expire_after', '—'),
        f'{ret.get("min_count", "—")}/{ret.get("max_count", "—")}',
        pdata.get('next_execution', '—'),
    )
console.print(t)

---
## 8. Cron Expression Reference

Elasticsearch uses a **6-field Quartz cron** (not standard 5-field Unix cron):

```
┌─────────── second     (0–59)
│ ┌───────── minute     (0–59)
│ │ ┌─────── hour       (0–23)
│ │ │ ┌───── day of month (1–31, or ?)
│ │ │ │ ┌─── month      (1–12 or JAN–DEC)
│ │ │ │ │ ┌─ day of week  (1–7 or SUN–SAT, or ?)
│ │ │ │ │ │
* * * * * *
```

Use `?` for day-of-month or day-of-week (not both can have a value).

In [ ]:
heading('8. Common cron expressions')

examples = [
    ('0 0 * * * ?',       'Every hour'),
    ('0 30 1 * * ?',      'Daily at 01:30'),
    ('0 0 2 ? * MON',     'Every Monday at 02:00'),
    ('0 0 1 1 * ?',       'First day of every month at 01:00'),
    ('0 0/30 * * * ?',    'Every 30 minutes'),
    ('0 0 4 ? * MON-FRI', 'Weekdays at 04:00'),
]

t = Table('Cron expression', 'Meaning')
for expr, meaning in examples:
    t.add_row(expr, meaning)
console.print(t)

---
## 9. Delete an SLM Policy

In [ ]:
heading('9. Delete SLM policies')

for policy_id in ['training-hourly', 'training-daily', 'training-weekly']:
    try:
        es.slm.delete_lifecycle(policy_id=policy_id)
        success(f'Deleted policy: {policy_id}')
    except Exception as e:
        warn(f'Could not delete {policy_id}: {e}')

info('Deleting a policy does NOT delete the snapshots it created.')
info('You must delete snapshots separately via DELETE /_snapshot/{repo}/{snap}.')

## Summary

| Task | API |
|------|-----|
| Create policy | `PUT /_slm/policy/{id}` |
| Get policy | `GET /_slm/policy/{id}` |
| Execute now | `PUT /_slm/policy/{id}/_execute` |
| Run retention | `POST /_slm/_execute_retention` |
| Stats | `GET /_slm/stats` |
| Status | `GET /_slm/status` |
| Stop / Start | `POST /_slm/stop` / `POST /_slm/start` |
| Delete policy | `DELETE /_slm/policy/{id}` |

**Next:** [07_advanced_topics.ipynb](07_advanced_topics.ipynb)